# 🛰️ GFM Flood Observation – From Satellite Detection to Interactive 3D Map

### Introduction

The **Global Flood Monitoring (GFM)** system by the **Copernicus Emergency Management Service (CEMS)** provides near-real-time flood information derived from Sentinel-1 radar imagery.  
These products enable rapid assessment of flood extent and dynamics anywhere on Earth — independent of cloud cover or daylight.

This notebook demonstrates how to transform **GFM flood-extent time series** into a spatially explicit summary and visualize it interactively.

### Storyline

We start with **GFM ensemble flood-extent rasters** (`ensemble_flood_extent`), which represent flooded pixels (value > 0) across multiple Sentinel-1 acquisition dates.  
Each pixel indicates whether water was detected during a specific satellite pass.  
By stacking these time slices, we can count **how many times the satellite observed flooding** in each location during the selected period.

The workflow proceeds step by step:
1. **Load and mask** the GFM raster dataset (`ensemble_flood_extent`) to remove NoData pixels.  
2. **Aggregate over time** to calculate the total number of flood observations per pixel.  
3. **Polygonize** all areas with at least one detected flood (8-connectivity ensures continuous shapes).  
4. **Export** the result as a GeoJSON containing each flood polygon with an attribute  
   `GFM_observed_flood` = *number of flood observations by the satellite*.  
5. **Visualize** the polygons in a **3D interactive map** using `leafmap.maplibregl`,  
   with color and extrusion height representing the number of flood detections.  
6. **Automatically generate a legend** based on the maximum number of observed flood events.

### Purpose

This notebook provides a **compact end-to-end workflow** for analyzing Copernicus GFM data:
- Convert temporal flood rasters into interpretable vector layers.  
- Explore the spatial structure and persistence of observed floods.  
- Generate publication-ready or dashboard-ready interactive maps.

Such processing can be applied to any AOI, from local river basins to continental scales,  
supporting flood-risk assessment, post-event mapping, and exposure analysis.


## Load libraries

In [ ]:
# === IMPORTS ===
from shapely.geometry import box, shape, mapping
from pystac_client import Client
import odc.stac
import pyproj
import rioxarray
import xarray as xr
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import shapes
import os
import json, math
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, Rectangle
from datetime import date, timedelta

import leafmap.maplibregl as leafmap

## Draw an Area of Interest (AOI)

Use the map below to **draw a rectangle** around the area you want to analyze.  

⚠️ Rules for drawing:  
- Only rectangles are allowed (no polygons, lines or circles).  
- To keep things efficient, a **size limit** is enforced:  
  – the sum of width + height of the rectangle must be ≤ `MAX_SIZE` degrees.  
- If your selection is too large, you’ll see a warning and the AOI won’t be accepted.  

After drawing, the **selected bounding box** (min/max longitude/latitude) will be printed under the map.  
The default bounding box is shown in **green**, and your new selection will appear in **red**.

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

# Create map
from ipyleaflet import TileLayer
basemap_carto = TileLayer(url="https://gisco-services.ec.europa.eu/maps/wmts/OSMCartoComposite/EPSG3857/{z}/{x}/{y}.png", attribution="© OpenStreetMap contributors © European Commission - Eurostat/GISCO")

m = Map(basemap=basemap_carto, center=(48.5, 11.0), zoom=9)

# Accept only rectangle
draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#3388ff", "fillOpacity": 0.1}},
    polygon={}, circle={}, circlemarker={}, polyline={}
)

output = widgets.Output()

# limit sum (width + height)
MAX_SIZE = 2

# ---- Default bbox ----
default_bbox = (10.5,48.5,10.9,48.7746)  # (minx, miny, maxx, maxy)

# save to variabel
selected_bbox = {
    "minx": default_bbox[0],
    "miny": default_bbox[1],
    "maxx": default_bbox[2],
    "maxy": default_bbox[3]
}

# draw to map like rectangle
last_rect = Rectangle(
    bounds=((default_bbox[1], default_bbox[0]), (default_bbox[3], default_bbox[2])),
    color="green",
    fill_opacity=0.2
)
m.add_layer(last_rect)

with output:
    print("Default bounding box:", selected_bbox)

# ---- Callback for new bbox ----
def handle_draw(_, action, geo_json):
    global selected_bbox, last_rect

    # delete old box
    if last_rect in m.layers:
        m.remove_layer(last_rect)

    if geo_json["geometry"]["type"] == "Polygon":
        geom = shape(geo_json["geometry"])
        minx, miny, maxx, maxy = geom.bounds
        width = maxx - minx
        height = maxy - miny
        size = width + height

        with output:
            output.clear_output()
            if size > MAX_SIZE:
                print(f"⚠️ Bounding box is too big! "
                      f"Max (width+height): {MAX_SIZE}°, "
                      f"currently: {size:.3f}° (width {width:.3f}°, height {height:.3f}°)")
                selected_bbox = {}
            else:
                selected_bbox = {
                    "minx": minx,
                    "miny": miny,
                    "maxx": maxx,
                    "maxy": maxy
                }
                # draw new rectangle
                last_rect = Rectangle(
                    bounds=((miny, minx), (maxy, maxx)),
                    color="red",
                    fill_opacity=0.2
                )
                m.add_layer(last_rect)

                print("Selected bounding box:", selected_bbox)

draw_control.on_draw(handle_draw)
m.add_control(draw_control)

display(m, output)


## Pick a date range

Flood layers are stored **day by day**.  
To analyze a specific flood, you need to know **when the event happened**.  

- Select the start and end dates that cover the flood period.  
- The notebook will load all available GFM flood maps in this range. 

In [ ]:
# DatePicker start and end
start_picker = widgets.DatePicker(
    description="From:",
    value=date(2024, 5, 27), 
    disabled=False
)

end_picker = widgets.DatePicker(
    description="To:",
    value=date(2024, 6, 17),  
    disabled=False
)

output = widgets.Output()
selected_dates = []

def update_range(change=None):
    global selected_dates
    if start_picker.value and end_picker.value:
        start = start_picker.value
        end = end_picker.value
        if start > end:
            with output:
                output.clear_output()
                print("⚠️ The start date must be earlier than or equal to the end date..")
            return
        selected_dates = [(start + timedelta(days=i)).isoformat() 
                          for i in range((end - start).days + 1)]
        with output:
            output.clear_output()
            print("Selected date:", selected_dates)

# Follow pickers
start_picker.observe(update_range, names="value")
end_picker.observe(update_range, names="value")

# ✅ default time range start
update_range()

display(widgets.HBox([start_picker, end_picker]), output)


## Prepare AOI geometry and time range 🗂️

Before we can query flood data, we need two things:

1. **Area of Interest (AOI):**  
   The rectangle you drew on the map is converted into a Shapely geometry object.  
   – This gives us exact bounding box coordinates (min/max longitude & latitude).  

2. **Time range:**  
   The dates you selected earlier are combined into a start and end date.  
   – This tells the STAC API which flood images to load.  

Both the AOI bounds and the selected time range are printed below for verification.

In [ ]:
from shapely.geometry import box

STAC_URL = "https://stac.eodc.eu/api/v1"
COLLECTION_ID = "GFM"

# convert selected_bbox to a Shapely box
if selected_bbox:
    aoi_geometry = box(
        selected_bbox["minx"],
        selected_bbox["miny"],
        selected_bbox["maxx"],
        selected_bbox["maxy"]
    )
else:
    aoi_geometry = None  # if nothing has been drawn yet

# time range from widgets (list of days in ISO format)
if 'selected_dates' in globals() and selected_dates:
    time_range = (selected_dates[0], selected_dates[-1])  # first and last date
else:
    time_range = None  # if no date range selected yet

print("AOI geometry:", aoi_geometry)
if aoi_geometry:
    print("AOI bounds:", aoi_geometry.bounds)
print("Time range:", time_range)


## STAC search + load

In this step we connect to the **STAC API** (SpatioTemporal Asset Catalog) at  
[EODC](https://stac.eodc.eu/api/v1) and request flood data from the **GFM collection**.

Process:

1. **Search the catalog:**  
   – Uses your AOI geometry (rectangle) and the selected date range.  
   – Returns all matching flood data “items” (satellite-derived rasters).  
   – The notebook prints how many items were found and how long the search took.   

2. **Load rasters:**  
   – Data are loaded lazily via `odc-stac` (efficient, chunked).  
   – The source CRS (coordinate reference system) and resolution are read from metadata.  

3. **Reproject to EPSG:4326 (WGS84):**  
   – Ensures all downstream steps use a common geographic coordinate system.  
   – This makes overlays with web maps (Folium, ipyleaflet) work correctly.  

✅ Output:  
– List of STAC items found.  
– Loaded rasters in memory (aligned to your AOI, time range, and selected bands).  


In [ ]:
# === STAC QUERY ===
client = Client.open(STAC_URL)
search = client.search(
    collections=[COLLECTION_ID],
    intersects=mapping(aoi_geometry),
    datetime=f"{time_range[0]}/{time_range[1]}"
)
items = search.item_collection()
print(f"🔍 Found {len(items)} items.")
if len(items) == 0:
    raise RuntimeError("No STAC items found.")

# CRS & resolution
crs = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"])
res = items[0].properties.get("gsd", 20.0)
print("Source CRS:", crs)
print("Resolution:", res)

# === LOAD DATA ===
xx = odc.stac.load(
    items,
    crs=crs,
    bbox=aoi_geometry.bounds,
    bands=["ensemble_flood_extent"],
    resolution=res,
    dtype="uint8",
    fail_on_error=False,
)
xx = xx.rio.reproject("EPSG:4326")

### 🔹 Flood extent polygonization (observed flood duration)

This code converts a time-series flood raster (`ensemble_flood_extent`) into polygons representing the number of observed floods in time range:

1. **Mask NoData values** and create a boolean raster of flooded / non-flooded cells.  
2. **Sum across the time dimension** to get total flooded observation per pixel.  
3. **Polygonize** areas with at least one flooded observation.  
4. **Build a GeoDataFrame** with attributes:
   - `geometry` — flood polygon  
   - `GFM_observed_flood` — number of flooded observation  
   - `time_range` — date range of observation  
6. **Export** to `observed_flood.geojson` (EPSG:4326).  
7. **Print summary** of polygon counts.

Output:  
- File: `observed_flood.geojson`  
- Field: `GFM_observed_flood` (integer, observed floods in time range)  


In [ ]:
# source
flood = xx["ensemble_flood_extent"]  # (time, y, x), uint8, nodata=255

# IMPORTANT: After reprojection, NoData (255) might become 254 -> exclude BOTH.
# Value 0 = satellite observed but NO flood (dry ground) -> this STILL counts as a pass.
#
# COUNT IN DAYS, NOT TILE-LAYERS:
# GFM tiles one acquisition into several STAC items, and this cube is NOT grouped by
# solar day, so every tile is its own time slice. A pixel covered by 2 overlapping tiles
# on the SAME day must count as ONE pass, not two. So we collapse same-day slices with
# .any() BEFORE summing -> both metrics become a number of distinct DAYS.

vals = flood.values
if hasattr(vals, "compute"):          # dask -> numpy
    vals = vals.compute()
vals = np.asarray(vals)               # (T, y, x)

day_str = np.array([str(t)[:10] for t in flood["time"].values])  # 'YYYY-MM-DD' per slice

valid_t = (vals < 254)                # pixel observed that slice (dry 0 OR flood)
flood_t = (vals > 0) & (vals < 254)   # flood that slice

H, W = vals.shape[1:]
satellite_passes = np.zeros((H, W), dtype="uint16")  # DAYS the satellite observed the pixel
flood_sum        = np.zeros((H, W), dtype="uint16")  # DAYS a flood was captured at the pixel
for d in np.unique(day_str):
    sel = (day_str == d)
    satellite_passes += valid_t[sel].any(axis=0).astype("uint16")   # 1 if ANY tile saw the pixel that day
    flood_sum        += flood_t[sel].any(axis=0).astype("uint16")   # 1 if ANY tile flooded the pixel that day

print(f"Acquisition days in range: {len(np.unique(day_str))} "
      f"| max satellite passes over a pixel: {int(satellite_passes.max())} days "
      f"| max flood days over a pixel: {int(flood_sum.max())} days")

# arrays are already numpy (days); take spatial transform from the source DataArray
arr = np.ascontiguousarray(flood_sum)
passes_arr = np.ascontiguousarray(satellite_passes)
transform = flood.rio.transform()

# polygonize only where floods > 0 (8-connectivity)
mask = arr > 0
geoms = list(shapes(arr, mask=mask, transform=transform, connectivity=8))

# build GeoDataFrame with attribute including satellite_passes
records = []
for geom, val in geoms:
    if val > 0:
        poly = shape(geom)
        # Get representative point to sample satellite_passes raster
        centroid = poly.representative_point()
        # Convert centroid coords to pixel indices
        inv_transform = ~transform
        col, row = inv_transform * (centroid.x, centroid.y)
        col, row = int(round(col)), int(round(row))
        # Clamp to array bounds
        row = max(0, min(row, passes_arr.shape[0] - 1))
        col = max(0, min(col, passes_arr.shape[1] - 1))
        passes_val = int(passes_arr[row, col])
        records.append({
            "geometry": poly,
            "GFM_observed_flood": int(val),   # DAYS with a flood at this pixel
            "satellite_passes": passes_val    # DAYS the satellite observed this pixel
        })

gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
gdf["time_range"] = f"{time_range[0]} → {time_range[1]}"

# (optional) small geometry simplification based on resolution, to reduce file size
try:
    res_x, res_y = map(abs, flood.rio.resolution())
    tol = 0.5 * min(res_x, res_y)
except Exception:
    pass

# export to GeoJSON
out_file = "observed_flood.geojson"
gdf.to_file(out_file, driver="GeoJSON", COORDINATE_PRECISION=6)
print(f"✅ Saved {out_file} | features: {len(gdf)}")

# quick check of value distribution (how many polygons with 1 day, 2 days, …)
print(gdf["GFM_observed_flood"].value_counts().sort_index().head(20))
print(f"\nFlood days range:              {gdf['GFM_observed_flood'].min()} – {gdf['GFM_observed_flood'].max()}")
print(f"Satellite passes (days) range: {gdf['satellite_passes'].min()} – {gdf['satellite_passes'].max()}")


In [ ]:
from IPython.display import HTML
HTML("""
<style>
.maplibregl-popup-content { 
  color: #111 !important; 
  background: #fff !important; 
}
.maplibregl-popup-tip { 
  border-top-color: #fff !important; 
}
.maplibregl-popup-close-button { 
  color: #111 !important;
}
</style>
""")


### Dynamic 3D flood visualization

This code visualizes the **observed flood polygons** in 3D using `leafmap.maplibregl`:

1. **Inputs**  
   - `observed_flood.geojson` — polygons with attribute `GFM_observed_flood`  
   - Optional `aoi_geometry` for map centering; otherwise a fallback AOI is used.

2. **Basemap**  
   Uses **Esri World Imagery** as raster background.

3. **Dynamic legend**  
   - GeoJSON is read, all `GFM_observed_flood` values are scanned.  
   - The **maximum flood frequency** is detected to set legend limits.  
   - Colors use a 5-step blue scale (`#9ecae1` → `#08519c`).  
   - Legend dynamically adjusts (e.g. “1 time”, “2 times”, … “5+ times”).

4. **3D rendering**  
   - Each polygon is drawn as a **fill-extrusion** layer.  
   - Extrusion height = `GFM_observed_flood × 6 m` (configurable).  
   - Opacity = 0.85 for semi-transparent visualization.

5. **Legend and camera control**  
   - Title reflects date range from `start_picker` → `end_picker` if available.  
   - Map automatically tilts (`pitch = 45°`) and rotates (`bearing = 15°`) for 3D effect.

**Result:**  
Interactive 3D map showing flood intensity (number of flooded days) with an automatically generated legend and adaptive color ramp.  


In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

# Dynamic legend + colors based on the maximum value in GFM_observed_flood
import os, json
import leafmap.maplibregl as leafmap

# ==== INPUTS ====
url = "observed_flood.geojson"      # GeoJSON with attribute 'GFM_observed_flood'
ATTR = "GFM_observed_flood"

# ---- AOI (area of interest); safe fallback if not defined earlier
try:
    aoi_geometry
except NameError:
    from shapely.geometry import box
    aoi_geometry = box(10.70, 48.28, 10.98, 48.46)  # Augsburg fallback

minx, miny, maxx, maxy = aoi_geometry.bounds
lon_c = (minx + maxx) / 2
lat_c = (miny + maxy) / 2

# ---- Esri World Imagery basemap (no DEM, no sky)
style_esri = {
    "version": 8,
    "sources": {
        "esri": {
            "type": "raster",
            "tiles": [
                "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
            ],
            "tileSize": 256,
            "attribution": "Source: Esri — World Imagery © Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        }
    },
    "layers": [
        {"id": "esri-ortho", "type": "raster", "source": "esri"}
    ]
}

m = leafmap.Map(
    center=[lat_c, lon_c],  # leafmap maplibregl expects [lat, lon]
    zoom=12,
    pitch=0,
    bearing=0,
    style=style_esri,
)

# ==== DYNAMIC LEGEND AND MAX VALUE DETECTION ====
if not os.path.exists(url):
    print(f"⚠️ File '{url}' not found – only basemap will be displayed.")
else:
    # Load GeoJSON and extract values
    with open(url, "r", encoding="utf-8") as f:
        gj = json.load(f)

    vals = []
    for feat in gj.get("features", []):
        props = feat.get("properties", {})
        v = props.get(ATTR, None)
        try:
            if isinstance(v, str):
                v = float(v.strip())
            v = float(v)
            if v == int(v):
                v = int(v)
            vals.append(v)
        except Exception:
            continue

    # Ensure max is at least 1
    vals = [v for v in vals if v is not None]
    max_val = int(max(vals)) if vals else 1
    if max_val < 1:
        max_val = 1

    # Color palette for 1..5 (if max>5, last color covers 5+)
    palette = ["#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"]

    # ---- MATCH expression for discrete colors
    color_expr = ["match", ["to-number", ["get", ATTR]]]
    legend_dict = {}

    if max_val <= 5:
        for i in range(1, max_val + 1):
            color_expr.extend([i, palette[i - 1]])
            legend_dict[f"{i} times" if i > 1 else "1 time"] = palette[i - 1]
        # default color (if value outside range)
        color_expr.append(palette[0])
    else:
        for i in range(1, 5):
            color_expr.extend([i, palette[i - 1]])
            legend_dict[f"{i} times" if i > 1 else "1 time"] = palette[i - 1]
        # all >=5 values into last color
        color_expr.extend([5, palette[4]])
        color_expr.append(palette[4])
        legend_dict["5+ times"] = palette[4]

    # ---- Extrusion height
    val_num = ["coalesce", ["to-number", ["get", ATTR]], 0]
    height_expr = ["*", val_num, 6]  # adjust multiplier if needed

    paint_fill = {
        "fill-extrusion-color": color_expr,
        "fill-extrusion-height": height_expr,
        "fill-extrusion-opacity": 0.85,
    }

    # Add flood layer
    m.add_geojson(
        url,
        layer_type="fill-extrusion",
        paint=paint_fill,
        name="Observed flood extent",
        fit_bounds=False,
    )

    # ==== POPUP: show satellite_passes and GFM_observed_flood on click ====
    # Both attributes are counted in DAYS (see polygonization cell).
    popup_template = (
        "<div style='font-family: sans-serif; font-size: 13px; line-height: 1.6; white-space: nowrap;'>"
        "<b>🌊 Flood captured:</b> {{ GFM_observed_flood }} days<br>"
        "<b>🛰️ Satellite passes:</b> {{ satellite_passes }} days<br>"
        "<b style='white-space: nowrap;'>📅 Period:</b> <span style='white-space: nowrap;'>{{ time_range }}</span>"
        "</div>"
    )
    m.add_popup("Observed flood extent", template=popup_template)

    # ==== LEGEND TITLE (dynamic based on widgets, if available) ====
    try:
        start_date = start_picker.value
        end_date = end_picker.value
        if start_date and end_date:
            title = f"Observed floods by satellite<br>{start_date.isoformat()} → {end_date.isoformat()}"
        else:
            title = "Observed floods by satellite<br>(no range selected)"
    except NameError:
        title = "Observed floods by satellite<br>(no range selected)"

    m.add_legend(title=title, legend_dict=legend_dict, position="bottom-right")

# Camera (slight tilt for effect)
if hasattr(m, "fly_to"):
    m.fly_to(lon=lon_c, lat=lat_c, zoom=12, pitch=45, bearing=15, duration=4500)
else:
    m.fit_bounds([[minx, miny], [maxx, maxy]])
    if hasattr(m, "set_pitch"):
        m.set_pitch(45)

# ==== INFO PANEL: statistics + calendar visualization ====
import ipywidgets as widgets
from IPython.display import display
from datetime import date as dt_date, timedelta as dt_timedelta
from collections import OrderedDict
import calendar as _calmod

# --- STAC records per date (item EXISTS) — NOT the same as a real observation.
#     A GFM tile can intersect the AOI while the AOI itself is 100% NoData ("hole"). ---
try:
    image_dates = sorted(set(
        item.datetime.strftime("%Y-%m-%d") if item.datetime else None
        for item in items
    ) - {None})
    n_images = len(items)
except NameError:
    image_dates = []
    n_images = 0

# --- Classify days by REAL valid data over the AOI, not by item existence.
#     value 0 = observed & dry, 1..253 = flood, 254/255 = NoData (not observed). ---
flood_dates_set, observed_dates_set = set(), set()
try:
    flood_da = xx["ensemble_flood_extent"]
    for t_idx in range(flood_da.sizes["time"]):
        t_date = str(flood_da.time.values[t_idx])[:10]
        slice_data = flood_da.isel(time=t_idx).values
        if hasattr(slice_data, "compute"):
            slice_data = slice_data.compute()
        if (slice_data < 254).any():                        # >=1 observed pixel that day
            observed_dates_set.add(t_date)
        if ((slice_data > 0) & (slice_data < 254)).any():   # >=1 flooded pixel that day
            flood_dates_set.add(t_date)
except Exception:
    pass

# Real observation dates = days the satellite actually captured the AOI (dry OR flood)
n_observation_dates = len(observed_dates_set)

# Build the full date range from the selected period
try:
    range_start = start_picker.value
    range_end = end_picker.value
except NameError:
    range_start = dt_date.fromisoformat(time_range[0]) if time_range else dt_date(2024, 1, 27)
    range_end = dt_date.fromisoformat(time_range[1]) if time_range else dt_date(2024, 2, 17)

all_dates_in_range = []
d = range_start
while d <= range_end:
    all_dates_in_range.append(d.isoformat())
    d += dt_timedelta(days=1)

# Classify each date: flood / observed-dry / no-data (hole OR no item)
_flood_d, _dry_d, _gap_d = [], [], []
_cal_days = []

for ds in all_dates_in_range:
    d_obj = dt_date.fromisoformat(ds)
    tag = f"{d_obj.day}.{d_obj.month}"
    if ds in flood_dates_set:
        _flood_d.append(tag)
        _cal_days.append((d_obj, "flood"))
    elif ds in observed_dates_set:
        _dry_d.append(tag)
        _cal_days.append((d_obj, "dry"))
    else:
        _gap_d.append(tag)
        _cal_days.append((d_obj, "gap"))

# --- Build calendar HTML (same style as Global_flood_monitoring_time_series) ---
_GAP_BG = "repeating-linear-gradient(45deg,#efefef,#efefef 3px,#dcdcdc 3px,#dcdcdc 6px)"

def _cal_cell_style(cat):
    if cat == "flood":
        return "background:#2471a3;color:#fff;"
    if cat == "dry":
        return "background:#d6eaf8;color:#1b4f72;"
    if cat == "gap":
        return "background:" + _GAP_BG + ";color:#999;"
    return "background:transparent;color:#ccc;"

def _build_calendar_html(cal_days):
    if not cal_days:
        return '<i style="color:#888;">No calendar data</i>'
    cells = []
    prev_mo = None
    for _dd, _cat in cal_days:
        if _dd.month != prev_mo:
            if prev_mo is not None:
                cells.append('<div style="width:4px;"></div>')
            _lbl = f"{_calmod.month_abbr[_dd.month]}"
            cells.append(
                '<div style="font-weight:bold;font-size:9px;color:#555;'
                'display:flex;align-items:center;'
                'padding:0 3px;white-space:nowrap;">' + _lbl + '</div>'
            )
            prev_mo = _dd.month
        if _cat == "flood":
            _tip = f"{_dd.isoformat()} - flood detected"
        elif _cat == "dry":
            _tip = f"{_dd.isoformat()} - observed, no flood"
        else:
            _tip = f"{_dd.isoformat()} - no satellite observation"
        _style = _cal_cell_style(_cat)
        cells.append(
            f'<div title="{_tip}" style="{_style}border-radius:3px;text-align:center;'
            f'min-width:18px;padding:2px 1px;font-size:10px;">{_dd.day}</div>'
        )
    return ('<div style="display:flex;flex-wrap:nowrap;align-items:center;gap:1px;">'
            + "".join(cells) + '</div>')

def _cal_swatch(bg, border=""):
    _b = ("border:" + border + ";") if border else ""
    return ('<span style="display:inline-block;width:12px;height:12px;background:' + bg + ';'
            + _b + 'border-radius:3px;vertical-align:middle;margin-right:3px;"></span>')

_legend = (
    '<div style="margin-bottom:6px;font-size:12px;">'
    + _cal_swatch("#2471a3") + f'Flood ({len(_flood_d)}) &nbsp; '
    + _cal_swatch("#d6eaf8", "1px solid #aed6f1") + f'No flood ({len(_dry_d)}) &nbsp; '
    + _cal_swatch(_GAP_BG) + f'No satellite ({len(_gap_d)})'
    + '</div>'
)

info_html = (
    '<div style="margin-top: 10px; padding: 10px; background: #f8f9fa; '
    'border: 1px solid #dee2e6; border-radius: 6px; font-family: sans-serif; font-size: 13px; '
    'line-height: 1.6;">'
    '<div style="margin-bottom: 6px;">'
    f'<b>📅 Observation dates:</b> {n_observation_dates}'
    f'<span style="color:#888;"> &nbsp;({n_images} STAC records)</span>'
    + '</div>'
    + _legend
    + '<div style="overflow-x:auto;white-space:nowrap;">'
    + _build_calendar_html(_cal_days)
    + '</div>'
    + '</div>'
)

info_widget = widgets.HTML(value=info_html)

display(m)
display(info_widget)
